## Previs?o de Falha

Modelo para a predi??o da probabilidade de falha de dispositivos, a fim de reduzir custos ao antecipar o seu reparo.

Para isso, ap?s tratamento dos dados, foi gerado um modelo de floresta rand?mica para esta predi??o. Atrav?s de seus resultados, obtemos a resposta para a seguinte pergunta: Podemos realmente reduzir custos com este modelo de ML?

Resposta: **Sim**. O modelo gera aproximadamente **41,3%** de saving no conjunto de teste estudado, em compara??o ? situa??o na qual n?o ocorre a predi??o da falha dos dispositivos.

#### 1 - Importa??o dos Dados

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import joblib

In [ ]:
df_raw = pd.read_csv('data/full_devices.csv', sep=',', encoding='latin')
df_raw.head(5)

#### 2 - Pr?-processamento dos Dados

Verifica-se campos faltantes ou a presen?a de linhas duplicadas. Al?m disso, ? realizado um rebalanceamento das classes, de forma a torna-las mais homog?neas.

In [ ]:
df = df_raw.copy()
df.isna().sum()

In [ ]:
df.loc[df.duplicated()]

In [ ]:
df = df.drop_duplicates()
len(df.loc[df.duplicated()])

In [ ]:
X = df.drop(['date','device', 'failure'], axis=1)
y = df['failure']

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X,y)
print(len(X_resampled))
print(len(y_resampled))

In [ ]:
y_resampled.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

#### 3 - Treinamento do Modelo

Dentre os modelos, foi escolhido um modelo de ensemble do tipo Floresta Rand?mica

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


#### 4 - An?lise dos Resultados

Foram medidas a acur?cia, o F1-Score e a matriz de confus?o para determinar n?o somente a efic?cia do modelo, mas tamb?m a sua viabiliza??o para o neg?cio de acordo com o problema apresentado (1.0 para falhas e 0.5 para reparos)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
classif_report = classification_report(y_test, y_pred)

print(f'Acur?cia: {accuracy}')
print('Relat?rio de Classifica??o:\n', classif_report)

In [ ]:
confusion_matrix = confusion_matrix(y_test, y_pred)
confusion_matrix

Falsos positivos e Verdadeiros negativos foram considerados como Reparos, enquanto que falsos negativos foram tomados como Falhas

In [ ]:
fp = confusion_matrix[1,0] #Reparo: R$0.5
tn = confusion_matrix[1,1] #Reparo: R$0.5
fn = confusion_matrix[0,1] #Falha: R$1.0

In [ ]:
preco_original = y_test.to_list().count(1)
preco_predicao = fp * 0.5 + tn * 0.5 + fn * 1

print(f'Pre?o total dos dispositivos com falha: R${round(preco_original,2)}')
print(f'Pre?o dos dispositivos ap?s reparo via predi??o do modelo: R${round(preco_predicao, 2)}')
print(f'Porcentagem Salva: {round(100 - preco_predicao/preco_original*100,2)}%')

#### 6 - Exporta??o do Modelo

Modelo foi serializado para aplica??o futura

In [ ]:
trained_model = joblib.dump(model,'models/failure_devices_model.pkl')
trained_scaler = joblib.dump(scaler,'models/failure_devices_scaler.pkl')

#### 7 - Utiliza??o do Modelo

Exemplo de aplica??o do modelo utilizando 2 dispositivos extra?dos do dataset (e que n?o foram utilizados no treino ou no teste), para fins de demonstra??o

In [ ]:
def prepare_data(path):
    df_test = pd.read_csv(path, sep=',', encoding='latin')
    data = df_test.drop(['date', 'device', 'failure'], axis=1)
    return df_test, data

def predict_failure(scaler, model, data):

    if len(data) == 1:
        data = data.reshape(1, -1)

    dados_producao_normalizados = scaler.transform(data)
    probabilidades_falha = model.predict_proba(dados_producao_normalizados)[:, 1]

    return probabilidades_falha

In [ ]:
load_scaler = joblib.load('models/failure_devices_scaler.pkl')
load_model = joblib.load('models/failure_devices_model.pkl')

df_test = pd.read_csv('data/example_devices.csv', sep=',', encoding='latin')
data = df_test.drop(['date', 'device', 'failure'], axis=1)

failures_prob = predict_failure(load_scaler, load_model, data)
failures = pd.DataFrame({'Prob. Failures (%)': failures_prob})

df_report = pd.concat([df_test,failures], axis=1)
df_report

In [ ]:
df_report.to_excel('output/failure_report.xlsx', index=False)

#### 8 - Trabalhos Futuros

A otimiza??o dos hiperpar?metros do modelo, a utiliza??o de valida??o cruzada, a cria??o de um conjunto de valida??o e o estudo das correla??es dos atributos utilizados s?o a??es que podem ser realizadas para aprimorar os resultados obtidos.

 A adapta??o do c?digo, como a obten??o de logs de treinamento e de predi??o e o tratamento das Exceptions que podem vir a ocorrer, bem como a modelagem para uma API e conteiniza??o, permitem a sua aplica??o em produ??o para ser inserida localmente ou em Cloud.